# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidrazabajwa49/flyrank-ml-internship-assignment-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Type: binary classification.**

Not clustering — clustering groups content by similarity, it doesn't tell a strategist which page to act on. Not ranking or scoring (yet) — both need a risk signal to rank or score *by*, and that signal is exactly what this week produces; ranking is a natural next step downstream of this, not a replacement for it. I want a hard decision this week: does this page go on the watch list, yes or no. That's a classification question, and it's the modeling family I already have the most reps in (churn, fraud, spam detection).

In [1]:
# Reasoning cell — no computation needed yet.
print("Task type: binary classification")

Task type: binary classification


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_down` = 1 if `trend_direction == 'down'`, else 0.

**This is a proxy, not an observed outcome.** `trend_direction` itself comes from a defined rule — a threshold on `trend_pct`, which compares the last-30-day window against the prior-30-day window. It isn't an independently labeled business event (like a customer canceling); it's a rule's output, and I'm predicting that rule's output from a *different* set of features. That distinction matters for what I can claim later: I'm not forecasting a page's actual future traffic, I'm classifying which side of an existing rule a page falls on.

**A real limitation, stated up front:** this is a single 90-day snapshot, not a time series. I don't have a genuine 'future' period to test forecasting against. So the most honest use of this classifier isn't 'predicting what a page will do next' — it's a diagnostic: which content/context attributes correlate with landing in the 'down' bucket, learned from established content, and most usefully applied to *cold-start* pages that don't have their own 90-day trend history yet.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_down"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution (is_down):")
print((df["is_down"].value_counts(normalize=True) * 100).round(1))

Target distribution (is_down):
is_down
1    54.2
0    45.8
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: macro-F1.**

The classes are close to balanced (54.2% / 45.8%), so plain accuracy sounds reasonable but hides a real failure mode: a model that always predicts 'down' never flags a healthy page, which is exactly the false-positive-avoidance the strategist's fixed review budget needs. Macro-F1 forces both classes — and both sides of the two-sided cost from last week — to count. Computed below: a lazy 'always predict down' baseline scores 70.3% F1 on the down class but 0% on not-down, for a macro-F1 of 35.2%. **'Good' means clearing that 35.2% floor by a wide, real margin** — not matching or barely beating it.

In [3]:
from sklearn.metrics import f1_score, precision_score, recall_score

y = df["is_down"]
always_down = pd.Series(1, index=df.index)  # the laziest possible baseline

print("Always-predict-'down' baseline:")
print("  precision(down):", round(precision_score(y, always_down, pos_label=1) * 100, 1))
print("  recall(down):   ", round(recall_score(y, always_down, pos_label=1) * 100, 1))
print("  f1(down):       ", round(f1_score(y, always_down, pos_label=1) * 100, 1))
print("  f1(not-down):   ", round(f1_score(y, always_down, pos_label=0, zero_division=0) * 100, 1))
print("  macro-F1:       ", round(f1_score(y, always_down, average="macro", zero_division=0) * 100, 1),
      "<- the floor any real classifier has to clear")

Always-predict-'down' baseline:
  precision(down): 54.2
  recall(down):    100.0
  f1(down):        70.3
  f1(not-down):    0.0
  macro-F1:        35.2 <- the floor any real classifier has to clear


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one piece of content (`content_id`), at its current snapshot.** The feature slice below is deliberately restricted to attributes that exist independently of the last-30d / prev-30d performance windows the `trend_direction` label is built from — carried over from last week's leakage check. `impression_tier` and `position_tier` are excluded even though they show large group-wise separation (checked in Section 5) precisely *because* a collapsing search position is close to circular with a page already being 'down' — not an independent signal.

In [4]:
safe_features = [
    "content_id", "client_id", "search_volume", "competition", "competition_level", "cpc",
    "content_type", "main_intent", "word_count", "word_count_tier", "char_count",
    "provider_used", "model_used", "content_age_days", "age_tier",
    "days_since_last_update", "freshness_tier",
]

model_df = df[safe_features + ["is_down"]].copy()
print("Unit-of-analysis dataframe shape:", model_df.shape)
model_df.head()

Unit-of-analysis dataframe shape: (30000, 18)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,word_count_tier,char_count,provider_used,model_used,content_age_days,age_tier,days_since_last_update,freshness_tier,is_down
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,2000-3500,20457.0,NaN,gemini-2.5-flash,187,181-365,20,0-30,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,2000-3500,15562.0,NaN,gemini-3-flash-preview,445,365+,25,0-30,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,3500+,23643.0,NaN,gemini-2.5-flash,141,91-180,20,0-30,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,NaN,463,365+,22,0-30,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,2000-3500,17469.0,NaN,gemini-3-flash-preview,263,181-365,14,0-30,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Checked every leakage-safe categorical column for how far it alone separates `is_down` (min-max spread across its groups, groups under 100 rows dropped as too thin to trust):

| column | spread |
|---|---|
| competition_level | 1.1 pt |
| main_intent | 2.8 pt |
| freshness_tier | 14.0 pt |
| content_type | 28.6 pt |
| word_count_tier | 39.0 pt |

`word_count_tier` is the strongest single lever — but it's also 25.7% missing, so a rule built on it alone silently fails on a quarter of the catalog. And no single column gets close enough to a clean cut to hand a strategist a one-line rule with a defensible false-positive rate. The real signal looks like it lives in *combinations* — a short, old, low-competition page behaves differently than a short, fresh, high-competition one — and writing that out as nested if/elif branches gets unmanageable fast, which is exactly the interaction-handling a model (even a plain decision tree, which I've already built from scratch) does automatically instead of by hand.

In [5]:
candidates = ["freshness_tier", "main_intent", "competition_level", "content_type", "word_count_tier"]
for col in candidates:
    sizes = df[col].value_counts()
    big = sizes[sizes >= 100].index
    g = df[df[col].isin(big)].groupby(col)["is_down"].mean() * 100
    spread = g.max() - g.min() if len(g) > 1 else 0
    print(f"{col:18s} spread={spread:5.1f}pt  (min={g.min():.1f}, max={g.max():.1f})")

missing_wc_tier = df["word_count_tier"].isna().mean() * 100
print(f"\nword_count_tier missing on {missing_wc_tier:.1f}% of rows — the strongest single lever, but not a complete one.")

freshness_tier     spread= 14.0pt  (min=47.1, max=61.1)
main_intent        spread=  2.8pt  (min=54.2, max=57.1)
competition_level  spread=  1.1pt  (min=55.6, max=56.7)
content_type       spread= 28.6pt  (min=28.7, max=57.2)
word_count_tier    spread= 39.0pt  (min=20.7, max=59.7)

word_count_tier missing on 25.7% of rows — the strongest single lever, but not a complete one.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.